In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import os
import transformers

/Users/keithatienza/Desktop/Academics/Modern Data Analytics/Horizon-Europe-MDA/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
raw_directory = '../data/raw'

In [3]:
project = pd.read_excel(raw_directory + "/project" + ".xlsx")
euroscivoc = pd.read_excel(raw_directory + "/euroSciVoc" + ".xlsx")


/Users/keithatienza/Desktop/Academics/Modern Data Analytics/Horizon-Europe-MDA/.venv/lib/python3.11/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/Users/keithatienza/Desktop/Academics/Modern Data Analytics/Horizon-Europe-MDA/.venv/lib/python3.11/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/Users/keithatienza/Desktop/Academics/Modern Data Analytics/Horizon-Europe-MDA/.venv/lib/python3.11/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [4]:
project.columns

Index(['id', 'acronym', 'status', 'title', 'startDate', 'endDate', 'totalCost',
       'ecMaxContribution', 'legalBasis', 'topics', 'ecSignatureDate',
       'frameworkProgramme', 'masterCall', 'subCall', 'fundingScheme',
       'nature', 'objective', 'contentUpdateDate', 'rcn', 'grantDoi'],
      dtype='object')

In [5]:
# Extract the second root from euroscivocpath for each project
# Merge project and euroscivoc on project id
merged = pd.merge(project, euroscivoc, left_on='id', right_on='projectID', how='left')

# Function to extract the second root from euroscivocpath
# euroscivocpath is expected to be a string like '/root1/root2/...'
def extract_second_root(path):
    if pd.isna(path):
        return None
    parts = path.split('/')
    # parts[0] is empty due to leading slash, so second root is parts[2] if it exists
    return parts[2] if len(parts) > 2 else None

# Group by project, aggregate all second roots as a list
extracts = (
    merged.groupby(['id', 'title', 'objective'])['euroSciVocPath']
    .apply(lambda paths: list({extract_second_root(p) for p in paths if extract_second_root(p) is not None}))
    .reset_index()
    .rename(columns={'id': 'project_id', 'euroSciVocPath': 'euroscivoc_extract'})
)

# Show the resulting dataframe
extracts.head()

,project_id,title,objective,euroscivoc_extract
0,101039048,Gaseous detectors for neutrino physics at the ...,The recent detection of the coherent elastic n...,"[chemical sciences, physical sciences]"
1,101039060,Tracing the Epipalaeolithic origins of plant m...,The transition from foraging to farming repres...,"[history and archaeology, earth and related en..."
2,101039066,Climate change impacts on trees reproduction a...,The capacity of future forests to support biod...,"[biological sciences, earth and related enviro..."
3,101039090,Machine-Assisted Teaching for Open-Ended Probl...,Computational thinking and problem solving ski...,"[computer and information sciences, educationa..."
4,101039098,Enhanced quantum resilience through twists,Quantum technology will revolutionize informat...,[physical sciences]


In [ ]:
from transformers import pipeline
from tqdm import tqdm

# Load zero-shot classifier
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

# Function to select the best label using zero-shot classification
def select_best_label(row):
    labels = row['euroscivoc_extract']
    # If no labels, return None
    if not labels or len(labels) == 0:
        return None
    # If only one label, return it directly
    if len(labels) == 1:
        return labels[0]
    # Concatenate title and objective for the sequence
    title = str(row['title']) if pd.notna(row['title']) else ''
    objective = str(row['objective']) if pd.notna(row['objective']) else ''
    sequence = (title + ' ' + objective).strip()
    # If sequence is empty, return None
    if not sequence:
        return None
    # Run zero-shot classification
    result = classifier(sequence, labels)
    # Return the label with the highest score
    return result['labels'][0]

# Apply to all rows with progress bar
tqdm.pandas()
extracts['euroscivoc_final'] = extracts.progress_apply(select_best_label, axis=1)

# Show the updated dataframe
extracts[['project_id', 'title', 'objective', 'euroscivoc_extract', 'euroscivoc_final']].head()

Device set to use mps:0
  0%|          | 12/15341 [00:11<4:04:27,  1.05it/s]



KeyboardInterrupt: 

In [8]:
extlabels = pd.read_csv("/Users/keithatienza/Desktop/Academics/Modern Data Analytics/Horizon-Europe-MDA/data/processed/extracts_with_labels.csv")

In [12]:
# Merge extlabels with project dataframe to add EC contribution, start date, end date, and total cost
# Assumes extlabels has a column 'project_id' matching 'id' in project
enriched = extlabels.merge(
    project[['id', 'ecMaxContribution', 'startDate', 'endDate', 'totalCost']],
    left_on='project_id', right_on='id', how='left'
)
# Drop the duplicate 'id' column from project
enriched = enriched.drop(columns=['id', 'objective', 'euroscivoc_extract'])
# Show the enriched dataframe
enriched.head()

,project_id,title,euroscivoc_final,ecMaxContribution,startDate,endDate,totalCost
0,101039048,Gaseous detectors for neutrino physics at the ...,physical sciences,1496205,2022-02-01,2027-01-31,1496205
1,101039060,Tracing the Epipalaeolithic origins of plant m...,history and archaeology,1499150,2023-09-01,2028-08-31,1499150
2,101039066,Climate change impacts on trees reproduction a...,"agriculture, forestry, and fisheries",1482050,2023-01-01,2027-12-31,1482050
3,101039090,Machine-Assisted Teaching for Open-Ended Probl...,educational sciences,1495000,2022-04-01,2027-03-31,1495000
4,101039098,Enhanced quantum resilience through twists,physical sciences,1458688,2023-03-01,2028-02-29,1458688


In [13]:
enriched.to_csv("../data/processed/topic_with_funding.csv", index=False)